# Лабораторная работа: классификация ЭКГ. Часть 2 — классификатор

В этой части вы обучите нейросетевой классификатор диагнозов на датасете, который собрали и экспортировали в первой части (`ecg-warehouse/notebooks/lab_dataset.ipynb`), оцените его на тестовой выборке и попробуете улучшить базовую модель.

Обучение построено на PyTorch Lightning и библиотеке `ecg_ml`: `WarehouseDatasetModule` загружает экспорт `ecg-warehouse`, `ClassifierLModule` реализует шаги обучения, валидации и теста.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного. Поэтому у сети по одному независимому выходу на класс, а не softmax по классам.

Ноутбук выполняется в окружении проекта `ecg-ml`: установите его командой `uv sync` в корне репозитория и выберите ядро из `.venv`.

In [1]:
import os
from pathlib import Path

# Пути в ноутбуке отсчитываются от корня репозитория, а Jupyter может запустить ноутбук из notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

### Параметры

- `VARIANT` — номер вашего варианта, тот же, что в первой части;
- `DATASET_PATH` — каталог, в который первая часть экспортировала датасет (путь относительно корня репозитория `ecg-ml`);
- `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS` — шаг обучения, размер батча и максимальное число эпох;
- `NUM_WORKERS` — число процессов, которые готовят батчи.

In [2]:
VARIANT = 6
DATASET_PATH = f"data/ptb-xl-variant-{VARIANT}"

LEARNING_RATE = 0.001
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 30

In [3]:
from functools import partial
import torch
from lightning import Trainer
from ecg_ml.data.warehouse import WarehouseDatasetModule
from ecg_ml.modules.classifier import ClassifierLModule

## 1. Данные

`WarehouseDatasetModule` читает `signals.npz` и `meta.csv` из экспорта. Тестовая выборка — часть `test` из разбиения хранилища. Валидационная выделяется из `train` тоже по пациентам, около 10 % пациентов; `random_state` фиксирует это разбиение, чтобы модели сравнивались на одних и тех же записях.

Классы и их порядок берутся из экспорта: `CLASSES` совпадёт со списком из первой части.

In [4]:
dataset = WarehouseDatasetModule(DATASET_PATH, random_state=0)
datamodule = dataset.to_datamodule(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

CLASSES = dataset.classes
CLASSES

['SR', 'AFIB', 'STACH', 'SBRAD']

## 2. Модель

В ячейке ниже — заготовка `ECGClassifier`.

**Задание.** Замените реализацию `ECGClassifier` в ячейке ниже собственной архитектурой нейросети для классификации ЭКГ. Сохраните имя класса и параметр `output_size`: модель создаётся в ячейке с `ClassifierLModule` как `ECGClassifier(output_size=len(CLASSES))`. Требования к модели:

- вход — батч сигналов формы `(batch, 12, 5000)`: 12 отведений по 5000 отсчётов;
- выход — логиты формы `(batch, len(CLASSES))`.

In [5]:
class ECGClassifier(torch.nn.Module):
    def __init__(self, output_size=4):
        super().__init__()

        self.features = torch.nn.Sequential(
            # Первый сверточный блок
            torch.nn.Conv1d(12, 32, kernel_size=15, stride=2, padding=7),
            torch.nn.BatchNorm1d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(kernel_size=2),

            # Второй сверточный блок
            torch.nn.Conv1d(32, 64, kernel_size=7, padding=3),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(kernel_size=2),

            # Третий сверточный блок
            torch.nn.Conv1d(64, 128, kernel_size=5, padding=2),
            torch.nn.BatchNorm1d(128),
            torch.nn.ReLU(),
            torch.nn.MaxPool1d(kernel_size=2),

            # Четвертый сверточный блок
            torch.nn.Conv1d(128, 256, kernel_size=3, padding=1),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),

            # Сжатие временной размерности
            torch.nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(256, output_size),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### Функция потерь и оптимизатор

- `BCEWithLogitsLoss` — бинарная кросс-энтропия, посчитанная независимо для каждого класса; подходит для многометочной задачи;
- `Adam` с шагом обучения `LEARNING_RATE`;
- `ReduceLROnPlateau` уменьшает шаг в 10 раз, если `val/loss` не улучшается 10 эпох подряд.

In [6]:
optimizer = partial(torch.optim.Adam, lr=LEARNING_RATE)

lr_scheduler = {
    "scheduler": partial(
        torch.optim.lr_scheduler.ReduceLROnPlateau,
        mode="min",
        factor=0.1,
        patience=10
    ),
    "monitor": "val/loss",
    "interval": "epoch",
    "frequency": 1,
}

criterion = torch.nn.BCEWithLogitsLoss()

`ClassifierLModule` объединяет сеть, функцию потерь, оптимизатор и метрики. На валидации и тесте считаются precision, recall и F1 для каждого класса и их macro-среднее.

In [7]:
classifier = ClassifierLModule(
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    criterion=criterion,
    network=ECGClassifier(output_size=len(CLASSES)),
    labels=CLASSES,
)

/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


## 3. Обучение

`Trainer` проводит валидацию после каждой эпохи и обучает не больше `EPOCHS` эпох в смешанной точности (`precision=16`). Метрики по эпохам сохраняются в `logs/lab/lightning_logs/version_N/metrics.csv`, веса — в `logs/lab/lightning_logs/version_N/checkpoints/`. Каталог `logs/` исключён из git.

In [8]:
trainer = Trainer(
    precision=16,
    check_val_every_n_epoch=1,
    min_epochs=1,
    max_epochs=EPOCHS,
    default_root_dir="logs/lab",
)


/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [li

In [9]:
trainer.fit(
    model=classifier,
    datamodule=datamodule,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                  ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network               │ ECGClassifier             │  161 K │ train │     0 │
│ 1 │ criterion             │ BCEWithLogitsLoss         │      0 │ train │     0 │
│ 2 │ val_metrics           │ MetricCollection          │      0 │ train │     0 │
│ 3 │ val_loss              │ MeanMetric                │      0 │ train │     0 │
│ 4 │ val_loss_best         │ MinMetric                 │      0 │ train │     0 │
│ 5 │ val_f1_macro_best     │ MaxMetric                 │      0 │ train │     0 │
│ 6 │ test_metrics          │ MetricCollection          │      0 │ train │     0 │
│ 7 │ test_confusion_matrix │ MultilabelConfusionMatrix │      0 │ train │     0 │
└───┴───────────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 161 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 161 K                                                                                                
Total estimated model params size (MB): 0.647                                                                      
Modules in train mode: 41                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/ksuny/study/lab4_ML/ML_HomeWork4_2/.venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:
21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

`Trainer.fit` stopped: `max_epochs=30` reached.


## 4. Оценка на тестовой выборке

`trainer.test` считает метрики на test для весов после последней эпохи и печатает `classification_report` из scikit-learn: precision, recall и F1 по каждому классу. Тестовую выборку используйте только для итоговой оценки, а модели и гиперпараметры сравнивайте по валидации.

In [10]:
trainer.test(
    model=classifier,
    datamodule=datamodule,
)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

precision    recall  f1-score   support

          SR       0.88      0.97      0.92      3341
        AFIB       0.88      0.86      0.87       335
       STACH       0.81      0.69      0.74       185
       SBRAD       0.69      0.30      0.42       126

   micro avg       0.87      0.92      0.90      3987
   macro avg       0.81      0.70      0.74      3987
weighted avg       0.87      0.92      0.89      3987
 samples avg       0.83      0.83      0.83      3987

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/f1/AFIB        │    0.8657616972923279     │
│       test/f1/SBRAD       │    0.41988950967788696    │
│        test/f1/SR         │    0.9211239218711853     │
│       test/f1/STACH       │    0.7426900863647461     │
│       test/f1/macro       │    0.7373663187026978     │
│    test/precision/AFIB    │           0.875           │
│   test/precision/SBRAD    │    0.6909090876579285     │
│     test/precision/SR     │    0.8798364996910095     │
│   test/precision/STACH    │     0.808917224407196     │
│   test/precision/macro    │    0.8136657476425171     │
│     test/recall/AFIB      │    0.8567163944244385     │
│     test/recall/SBRAD     │     0.30158731341362      │
│      test/recall/SR       │     0.96647709608078      │
│     test/recall/STACH     │    0.6864864826202393     │
│     test/recall/macro     │    0.7028168439865112     │
└───────────────────────────┴───────────────────────────┘

[{'test/f1/SR': 0.9211239218711853,
  'test/f1/AFIB': 0.8657616972923279,
  'test/f1/STACH': 0.7426900863647461,
  'test/f1/SBRAD': 0.41988950967788696,
  'test/f1/macro': 0.7373663187026978,
  'test/precision/SR': 0.8798364996910095,
  'test/precision/AFIB': 0.875,
  'test/precision/STACH': 0.808917224407196,
  'test/precision/SBRAD': 0.6909090876579285,
  'test/precision/macro': 0.8136657476425171,
  'test/recall/SR': 0.96647709608078,
  'test/recall/AFIB': 0.8567163944244385,
  'test/recall/STACH': 0.6864864826202393,
  'test/recall/SBRAD': 0.30158731341362,
  'test/recall/macro': 0.7028168439865112}]

## 5. Улучшение модели

Что можно попробовать для улучшения:

- подобрать `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS`;
- учесть дисбаланс классов: `torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(dataset.pos_weight))`;
- изменить архитектуру `ECGClassifier`: число и ширину слоёв, размер ядер свёрток, нормализацию, dropout.

In [11]:
criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        dataset.pos_weight,
        dtype=torch.float32
    )
)

## Сдача работы

Сохраните ноутбук вместе с результатами ячеек, закоммитьте его в свою ветку и откройте запросы на слияние в обоих форках, как описано в инструкции к лабораторной работе.